# Multiple MPC collection and fitting rounds per square band

Use a GPU runtime. The default is **three rounds per band and six starts per goal per round**, with the six original collection goals. Each rollout uses one-macro MPC from the start. The first round in each band uses the same start positions and planner seeds as the earlier wide MPC-only run. Further rounds sample fresh starts in the same band.

Collect a fixed batch, then fit for 500 updates on the new pairs and all retained pairs from prior rounds and bands. The next round collects with the cost just fitted. A single eligible pair triggers a fit. If the batch adds zero pairs, carry forward the incoming cost, using raw MSE until the first fit. Temporal labels come from successful rollouts without the 0.5-unit progress filter.

The default runs through Band 5. Set `LAST_BAND=8` to continue across the right room. Use `RESUME_RUN_ID` with the same round count and starts per goal to reuse completed batches. The tables and plots distinguish the cost that collected a round from the cost after fitting that round.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,shutil,signal,subprocess,sys,time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE=Path('/content/drive/MyDrive')
SOURCE_RUN=DRIVE/'datasets/curriculum_cost_adapter/radial_multigoal_ranking_v1/20261106T195304628603Z'
assert (SOURCE_RUN/'branches/multi/complete.json').is_file(), 'Set SOURCE_RUN to your completed multi-goal ranking run.'
RESUME_RUN_ID=None  # Set to the printed ID to resume an interrupted run.
ROUNDS_PER_BAND=3
STARTS_PER_GOAL=6
LAST_BAND=5                # Includes the previous failure band. Set to 8 to cover the right room.
PLOT_GOALS=[1]              # Collection goal indices for the trajectory panels
PLOT_BANDS=[2,3,4,5]
DRAW_COST_MAPS=True
HEATMAP_GOALS=[0,15]        # Evaluation goal indices for the raw vs adapted maps
RUN_ID=RESUME_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=DRIVE/'datasets/curriculum_cost_adapter/experience_mpc_repeated_half_square_v1'/RUN_ID
REPO=Path('/content/i-jepa-wm-experience-mpc-repeated')
REPO_URL='https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT='166cbf469d0a44bd5a14d2a46d91fa43f428479d'
DINO_ROOT=Path('/content/dino_wm')
DINO_COMMIT='0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE=DRIVE/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
WM_DRIVE=DRIVE/'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt'
ARTIFACTS=Path('/content/experience-one-macro-adapter-wide-artifacts')
ENCODER=ARTIFACTS/'encoder.pth.tar'
CHECKPOINT=ARTIFACTS/'world_model.pt'
ENCODER_SHA='3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
WM_SHA='b0f3ccfefe0534d9f51af2d0767619913f48d0c323ebb323f0bcd4b800206c3f'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def shell(command,cwd=None):
    command=list(map(str,command));print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout:print(line,end='',flush=True)
            code=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.terminate()
            raise
    if code:raise subprocess.CalledProcessError(code,command)

print('GPU:',torch.cuda.get_device_name())
print('Frozen ranking source:',SOURCE_RUN)
print('Run ID:',RUN_ID)
print('New experience curriculum results:',OUTPUT)

In [ ]:
if not REPO.exists():shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow','imageio','imageio-ffmpeg'])
if not DINO_ROOT.exists():shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ARTIFACTS.mkdir(parents=True,exist_ok=True)
for source,target,expected in ((ENCODER_DRIVE,ENCODER,ENCODER_SHA),(WM_DRIVE,CHECKPOINT,WM_SHA)):
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not target.exists() or sha256(target)!=expected:shutil.copy2(source,target)
    assert sha256(target)==expected, f'Unexpected checkpoint: {source}'

def command(operation, band=None):
    args=[sys.executable,'-m','src.mylonas_ijepa_wm.experience_curriculum',operation,
          '--source-run',SOURCE_RUN,'--output-dir',OUTPUT,'--world-model-checkpoint',CHECKPOINT,
          '--encoder-checkpoint',ENCODER,'--dino-wm-root',DINO_ROOT,'--device','cuda',
          '--starts-per-goal',STARTS_PER_GOAL,'--labels','temporal','--sampling','half-square','--planning-horizon',1,
          '--actions-per-plan',1,'--wide-square-bands',
          '--rounds-per-band',ROUNDS_PER_BAND]
    if band is not None:
        args+=['--band',band]
    return args

def run_band(band):
    started=time.monotonic()
    shell(command('stage',band),cwd=REPO)
    print('Band minutes:',round((time.monotonic()-started)/60,2))

## Collect and fit

Each completed round writes its own checkpoint and collection report. If a band cell is interrupted, rerun it with the same run ID. Collection and fitting resume in round order.

In [ ]:
assert ROUNDS_PER_BAND >= 2 and 1 <= LAST_BAND <= 8
for band in range(1,LAST_BAND+1):
    run_band(band)

## Compare collection rounds

A band entry occurs when the ball first reaches the previous square while staying on the right side of the wall. The table shows entries by action 5 and at any point, plus success, pairs, and optimizer updates. A blank entry count means there was no previous band. Each round uses a different batch of starts.

In [ ]:
from IPython.display import Image, display
import numpy as np
import pandas as pd

def read_json(path):
    return json.loads(path.read_text())

manifest=read_json(OUTPUT/'experiment.json')['manifest']
stages={}
for band in range(1,LAST_BAND+1):
    for number in range(1,ROUNDS_PER_BAND+1):
        path=OUTPUT/f'band_{band}_round_{number}'/'complete.json'
        if path.is_file():
            stages[band,number]=read_json(path)
if not stages:
    raise RuntimeError('Finish at least one collection round first.')

rows=[]
for (band,number),stage in stages.items():
    for row in stage['collection']:
        diag=row['boundary']
        rows.append(dict(band=band,round=number,goal=row['case']['goal_index'],
            success=row['success'],pairs=row['retained_pairs'],
            entered_five=diag['entered_by_five_actions'] if band>1 else np.nan,
            entered=diag['first_entry_step'] is not None if band>1 else np.nan,
            delta_five=diag['first_five_distance_change'],
            ended_farther=diag['final_distance_change']>0))
data=pd.DataFrame(rows)

def summarize(grouped):
    return grouped.agg(attempts=('success','size'),successes=('success','sum'),
        success_rate=('success','mean'),entries_by_5=('entered_five',lambda x:x.sum(min_count=1)),
        total_entries=('entered',lambda x:x.sum(min_count=1)),
        median_distance_change_5=('delta_five','median'),
        ended_farther_rate=('ended_farther','mean'),new_pairs=('pairs','sum')).round(3)

rounds=summarize(data.groupby(['band','round']))
rounds['cumulative_pairs']=rounds.new_pairs.cumsum()
rounds['updates']=[stages[key]['updates'] for key in rounds.index]
display(rounds)
by_goal=summarize(data.groupby(['band','round','goal']))
display(by_goal[by_goal.index.get_level_values('goal').isin(PLOT_GOALS)])
rounds.to_csv(OUTPUT/'round_summary.csv')
by_goal.to_csv(OUTPUT/'round_summary_by_goal.csv')

## Plot trajectories within each band

The panels show actual collection trajectories, using the incoming cost for each round. Green lines are successful paths. Solid red traces show the first five actions of failed paths, and dashed red segments join their starts and ends. White dots show retained training observations. The solid square is the inner boundary and the dashed square is the outer boundary.

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt

plot_dir=OUTPUT/'round_trajectories'
plot_dir.mkdir(exist_ok=True)
for band in PLOT_BANDS:
    for index in PLOT_GOALS:
        groups=[
            (number,[r for r in stages[band,number]['collection'] if r['case']['goal_index']==index])
            for number in range(1,ROUNDS_PER_BAND+1) if (band,number) in stages
        ]
        groups=[(number,group) for number,group in groups if group]
        if not groups:
            continue
        gx,gy=groups[0][1][0]['case']['goal']
        inner,outer=manifest['bands'][band-1]
        fig,axes=plt.subplots(1,len(groups),figsize=(4.8*len(groups),7),layout='constrained',squeeze=False)
        for ax,(number,group) in zip(axes[0],groups):
            for row in sorted(group,key=lambda r:r['success']):
                diag=row['boundary']
                xy=np.asarray(diag['positions'])
                color='#208445' if row['success'] else '#c83d46'
                if row['success']:
                    ax.plot(*xy.T,color=color,alpha=.68)
                    retained=np.asarray(row['retained_positions']).reshape(-1,2)
                    if len(retained):
                        ax.scatter(*retained.T,s=13,c='white',edgecolors=color,linewidths=.65,zorder=4)
                else:
                    ax.plot(*xy[:6].T,color=color,alpha=.52,linewidth=1.1)
                    ax.plot(*xy[[0,-1]].T,color=color,linestyle='--',alpha=.35)
                ax.scatter(*xy[0],s=32,facecolors='none',edgecolors=color,zorder=5)
                ax.scatter(*xy[-1],s=30,color=color,marker='x',zorder=5)
            for radius,style in ((inner,'-'),(outer,'--')):
                if radius:
                    ax.add_patch(Rectangle((gx-radius,gy-radius),2*radius,2*radius,
                                           fill=False,edgecolor='#555',linestyle=style,linewidth=1.1))
            ax.scatter(gx,gy,marker='*',s=120,c='black',zorder=6)
            successes=sum(r['success'] for r in group)
            entries=sum(r['boundary']['first_entry_step'] is not None for r in group)
            outcome=f'{successes}/{len(group)} success' if group else 'no attempts'
            if group and inner:
                outcome+=f' · {entries}/{len(group)} entered'
            ax.set(title=f'Round {number} · {outcome}',xlim=(35,60),ylim=(4,60),aspect='equal',
                   xlabel='Ball x',ylabel='Ball y')
        fig.suptitle(f'Collection goal {index} ({gx:.2f}, {gy:.2f}) · Band {band}')
        fig.legend(handles=[
            Line2D([],[],color='#208445',label='Success'),
            Line2D([],[],color='#c83d46',label='Failed first five'),
            Line2D([],[],color='#c83d46',ls='--',label='Failed start to end'),
            Line2D([],[],marker='o',mfc='white',mec='#208445',ls='none',label='Training observation'),
        ],loc='upper center',bbox_to_anchor=(.5,-.005),ncol=4,frameon=False)
        path=plot_dir/f'goal_{index}_band_{band}.png'
        fig.savefig(path,dpi=135,bbox_inches='tight')
        plt.close(fig)
        display(Image(filename=str(path)))

## Inspect changes to the cost

Each row shows the incoming cost and then the cost after each fit in that band. Amber outlines show the current square edges. A zero-update panel uses the carried-forward cost. Maps cover the entire right room through x = 60. Costs are divided by each map’s median so one color scale can show geometry changes. Titles report the original median MSE.

In [ ]:
if DRAW_COST_MAPS:
    finished=[b for b in range(1,LAST_BAND+1) if (b,ROUNDS_PER_BAND) in stages]
    if not finished:
        raise RuntimeError('Finish all rounds of at least one band to compare costs.')
    last=max(finished)
    shell(command('compare',last),cwd=REPO)
    report=read_json(OUTPUT/f'band_{last}_comparison.json')
    scores=pd.DataFrame([
        dict(cost=name,correct_order=np.mean([r['adapted_rank']['correct_fraction'] for r in rows[1:]]),
             uphill=np.mean([r['heldout_directional']['adapted_up_fraction'] for r in rows[1:]]))
        for name,rows in report['methods'].items()
    ])
    display(scores.round(3))
    scores.to_csv(OUTPUT/'round_cost_metrics.csv',index=False)
    grid=report['grid']
    shape=(len(grid['ys']),len(grid['xs']))
    plot_dir=OUTPUT/'heatmaps'
    plot_dir.mkdir(exist_ok=True)

    for goal_index in HEATMAP_GOALS:
        methods=report['methods']
        gx,gy=methods['raw'][goal_index]['goal']
        maps={name:np.asarray(rows[goal_index]['adapted_costs'])[grid['point_ids']].reshape(shape)
              for name,rows in methods.items() if name!='handcrafted'}
        medians={name:float(np.median(cost)) for name,cost in maps.items()}
        scaled={name:cost/max(medians[name],1e-9) for name,cost in maps.items()}
        vmax=max(float(cost.max()) for cost in scaled.values())
        cols=ROUNDS_PER_BAND+1
        fig,axes=plt.subplots(last,cols,figsize=(3.5*cols,4.6*last),layout='constrained',squeeze=False)
        for band in range(1,last+1):
            incoming='raw' if band==1 else f'band_{band-1}_round_{ROUNDS_PER_BAND}'
            names=[incoming,*[f'band_{band}_round_{r}' for r in range(1,cols)]]
            inner,outer=manifest['bands'][band-1]
            for number,(ax,name) in enumerate(zip(axes[band-1],names)):
                mesh=ax.pcolormesh(grid['xs'],grid['ys'],scaled[name],shading='nearest',
                                   cmap='viridis',vmin=0,vmax=vmax)
                for radius in (inner,outer):
                    if radius:
                        ax.add_patch(Rectangle((gx-radius,gy-radius),2*radius,2*radius,
                                               fill=False,edgecolor='#ffac3a',linestyle='--',linewidth=1.2))
                ax.scatter(gx,gy,marker='*',s=80,c='black',edgecolor='white',zorder=5)
                title='Incoming raw MSE' if name=='raw' else 'Incoming' if number==0 else f'After round {number}'
                if number and not report['checkpoints'][name]['updates']:
                    title+=' (carried forward)'
                ax.set(title=f'{title}\nMedian MSE {medians[name]:.3g}',xlim=(35,60),ylim=(4,60),
                       aspect='equal',xlabel='Ball x')
                if number==0:
                    ax.set_ylabel(f'Band {band} ({inner}–{outer})\nBall y')
        fig.colorbar(mesh,ax=axes.ravel().tolist(),shrink=.55,label='MSE divided by map median')
        fig.suptitle(f'Evaluation goal {goal_index} ({gx:.2f}, {gy:.2f})')
        path=plot_dir/f'goal_{goal_index}_rounds.png'
        fig.savefig(path,dpi=115,bbox_inches='tight')
        plt.close(fig)
        display(Image(filename=str(path)))